In [5]:
import json
import uuid
import random
import pickle
import os
from typing import Dict, Iterator, List, Optional, Set, Tuple



In [6]:

HalfFlight = Tuple[str, Tuple[int, int, int]]
Flight = Tuple[str, HalfFlight, HalfFlight]
Team = str


class Race:
    def __init__(self, team1: str, team2: str, race_num: int, flight: Optional[Flight] = None):
        self.team1 = team1
        self.team2 = team2
        self.race_num = race_num
        self.league = ""
        self.flight = flight

    def __str__(self):
        indent = "\t" * COLUMN_WIDTH * ((self.race_num - 1) % F)
        return f"{indent}{self.race_num}: {self.team1} vs {self.team2}"


def print_schedule(teams, races: List[Race]):
    total_raced = {t: 0 for t in teams}
    print("=" * 150)
    for race in races:
        print("\t", race)
        total_raced[race.team1] += 1
        total_raced[race.team2] += 1
        # print(total_raced)
        # print("min = ", min(total_raced.values()), "max = ", max(total_raced.values()))
        if all(total_raced[t] == total_raced[teams[0]] for t in teams):
            print("ALL EQUAL", total_raced[teams[0]])


# for pasting into google sheets or excel
def tsv_to_schedule(teams: List[str], flights: List[Flight], races: List[Race]) -> str:
    # header
    output = (
        "race\t"
        + "\t".join(f"{f1[0]}{f1[1]}\t{f2[0]}{f2[1]}" for f1, f2 in flights)
        + "\n"
    )

    # body
    total_raced = {t: 0 for t in teams}
    for race in races:
        total_raced[race.team1] += 1
        total_raced[race.team2] += 1

        output += f"{race.race_num}\t"
        flight_num = (race.race_num - 1) % len(flights)
        output += "\t" * (flight_num * 2)
        output += f"{race.team1}\t{race.team2}"
        # min_raced = min(total_raced.values())
        # max_raced = max(total_raced.values())
        # output += (
        #     "\t" * ((len(flights) - flight_num - 1) * 2)
        #     + f"\t{min_raced}\t{max_raced}\t{max_raced - min_raced}"
        # )
        output += "\n"

        # keep track of how many races each team has raced

        # if all teams have raced the same number of times, add a divider
        # if all(total_raced[t] == total_raced[teams[0]] for t in teams):
        #     output += "breakpoint\n"

    # footer
    return output



In [7]:
def results_json_to_tsv(path):
    with open(path, 'r') as f:
        results = json.load(f)

    results_str = ""
    for result in results:
        if result['league'] != 'quali': continue
        if result['finishtime'] is None: continue

        lteam = result['raceteam'][0]
        rteam = result['raceteam'][1]
        lwin = sum(lteam['result']) < sum(rteam['result'])

        results_str += f"{result['number']}\t"
        results_str += f"{lteam['team']['name']}\t"
        results_str += "Win\t" if lwin else "Loss\t"
        results_str += '\t'.join(map(str, lteam['result'])) + '\t'
        
        results_str += 'vs\t'

        results_str += '\t'.join(map(str, rteam['result'])) + '\t'
        results_str += "Loss\t" if lwin else "Win\t"
        results_str += f"{rteam['team']['name']}\n"

    print(results_str)


In [8]:
import os
from supabase import create_client, Client
from dotenv import load_dotenv

load_dotenv()

url: str = os.environ.get("SUPABASE_PROJECT_URL")
key: str = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")
supabase: Client = create_client(url, key)

In [9]:
from typing import Dict, Iterator, List, Set, Tuple

def tsv_to_schedule(path: str) -> Tuple[List[Race], List[Flight], Set[Team]]:
    lines = [l.replace('\n', '').split("\t") for l in open(path, "r")]
    flight_names = lines[0][1::2]
    lhalves = lines[1][1::2]
    rhalves = lines[1][2::2]
    flights = list(zip(flight_names, lhalves, rhalves))

    races = []
    teams = set()
    for line in lines[2:]:
        race_num = int(line[0])
        for flight, lteam, rteam in zip(flights, line[1::2], line[2::2]):
            if lteam != "":
                assert rteam != ""
                teams.add(lteam)
                teams.add(rteam)
                races.append(Race(lteam, rteam, race_num, flight))
                break

    return races, flights, teams


# Gets the id of each flight
# Returns mapping from flight name to id
def flight_ids() -> Dict[str, str]:
    fs = supabase.table("flight").select("*").execute().data
    print(fs)
    return {f["name"]: f["id"] for f in fs}


def team_ids() -> Dict[str, str]:
    ts = supabase.table("team").select("*").execute().data
    return {t["name"]: t["id"] for t in ts}


# Writes given schedule straight to the database
def write_schedule(
    schedule: List[Race], competition_id: str, start_num: int, league: str
):
    flights = flight_ids()
    teams = team_ids()
    rows = [
        {
            "number": i + start_num,
            "competition": competition_id,
            "flight": flights[race.flight[0]],
            "lteam": teams[race.team1],
            "rteam": teams[race.team2],
            "league": league,
        }
        for i, race in enumerate(schedule)
    ]
    supabase.table('race').insert(rows).execute()

# if __name__ == "__main__":
#     schedule, flights, teams = tsv_to_schedule("./schedule/topgun.tsv")  # tsv_to_schedule("./schedule/topgun.tsv")
#     # write_schedule(schedule, 'd909cc26-636f-4563-98ba-cef93e17c7fc', 1, 'quali')
#     team_races = {t: 0 for t in teams}
#     for race in schedule:
#         team_races[race.team1] += 1
#         team_races[race.team2] += 1
#     print(team_races)

{'Strathclyde': 42, 'Exeter': 42, 'Cambridge': 42, 'Southampton': 42, 'Loughborough': 42, 'Oxford': 42, 'Edinburgh': 42, 'Bristol': 42}


In [10]:
# iterates over a possible set of rounds
# if even number of teams, each round consists of each team playing against one other team
# if odd number of teams, each team plays two other teams
def generate_rounds(teams: List[str]) -> Iterator[List[Tuple[str, str]]]:
    ts = len(teams)

    if ts % 2 == 0:
        n = ts // 2
        team1s = teams[:n]
        team2s = teams[n:]
        # even number of teams
        print("even")
        for _ in range(n - 1):
            pairings = {t: set() for t in teams}
            for _ in range(2):
                for t1, t2 in zip(team1s, team2s):
                    pairings[t1].add(t2)
                    pairings[t2].add(t1)

                # rotate according to https://en.wikipedia.org/wiki/Round-robin_tournament
                team1s[1], team1s[2:], team2s[:-1], team2s[-1] = (
                    team2s[0],
                    team1s[1:-1],
                    team2s[1:],
                    team1s[-1],
                )

            # make these pairings into a full round
            # form a chain starting with l
            current = teams[0]
            round = []
            for i in range(ts):
                # get another team which matches with current
                other = pairings[current].pop()
                pairings[other].discard(current)
                # add to round
                race = (current, other) if i % 2 == 0 else (other, current)
                round.append(race)
                current = other
            yield round

        # add leftover round
        yield list(zip(team1s, team2s))
    else:
        # reorder teams
        n = ts // 2
        team1s = teams[:n]  # first half rounded down
        team2s = teams[n:]  # second half rounded up
        team2s[:-1] = team2s[:-1][::-1]

        for _ in range(n):
            # print("====")
            # print(team1s)
            # print(team2s)
            races = [(team2s[-1], team2s[0])]
            for t1, t2a, t2b in zip(team1s, team2s[:-1], team2s[1:]):
                races.append((t1, t2a))
                races.append((t1, t2b))
            yield races

            # rotates first n of each list, but keeps last team2s uneffected
            team1s[0], team1s[1:], team2s[-2], team2s[:-2] = (
                team2s[0],
                team1s[:-1],
                team1s[-1],
                team2s[1:-1],
            )

def check_schedule(races: List[Race], teams: List[str], flights: List[Flight], check_rr: bool = True):
    """
    criteria:
    - every team plays every other team exactly once
    - no team races more than 2 times in a row
    - races[i] is disjoint from races[i-1], ..., races[i-F] (otherwise they'd be busy)
    """
    f = len(flights)
    problems = []

    # - every team plays every other team exactly once
    pairings = {t: set(teams).difference(set([t])) for t in teams}
    for race in races:
        if (
            race.team2 not in pairings[race.team1]
            or race.team1 not in pairings[race.team2]
        ):
            problems.append(f"Team {race.team1} and {race.team2} raced more than once")

        pairings[race.team1].discard(race.team2)
        pairings[race.team2].discard(race.team1)
    if check_rr and not all(len(p) == 0 for p in pairings.values()):
        problems.append(f"Pairings remain: {pairings}")

    # - no team races more than 2 times in a row
    for i, race in enumerate(races):
        if i >= f + f:
            if (
                races[i - f].team1 == race.team1
                and races[i - f - f].team1 == race.team1
            ):
                # team1 raced in the two races before this one
                problems.append(f"Team {race.team1} has raced three times in a row")

            if (
                races[i - f].team2 == race.team2
                and races[i - f - f].team2 == race.team2
            ):
                # team2 raced in the two races before this one
                problems.append(f"Team {race.team2} has raced three times in a row")

    # - people aren't required to be in a boat too soon after being in another boat
    for i, race in enumerate(races):
        last_row = races[i - len(flights) : i]
        if len(last_row) > 0:
            busy_teams = {r.team1 for r in last_row}.union({r.team2 for r in last_row})
            if last_row[0].team1 == race.team1:
                busy_teams.remove(
                    race.team1
                )  # they're not preoccupied because already in correct boat
            if last_row[0].team2 == race.team2:
                busy_teams.remove(
                    race.team2
                )  # they're not preoccupied because already in correct boat

            if race.team1 in busy_teams:
                problems.append(
                    f"Team {race.team1} can't race on race {race.race_num} because they're busy"
                )
            if race.team2 in busy_teams:
                problems.append(
                    f"Team {race.team2} can't race on race {race.race_num} because they're busy"
                )

    if len(problems) > 0:
        print("problems:")
        print("\n".join(problems))
        raise ValueError(f"Schedule is invalid {len(problems)} problems")


def block_extend(
    races: List[Race],
    teams: List[str],
    flights: List[Flight],
    rounds: Dict[int, List[Tuple[str, str]]],
    best_schedule: List[Race],
) -> bool:
    """
    Trys to make a block which can be put on the end of races
    Returns True iff successful
    If unsuccessful, resets datastructures
    """
    if len(rounds) == 0:
        return True

    # try every block
    for round_num, r in list(rounds.items()):

        valid_block_found = False
        # print("new block")
        for _ in range(2 * len(r)):
            # form block
            block = [None] * len(r)
            i = 0
            flight = 0
            for lt, rt in r:
                block[i] = (lt, rt)
                i += len(flights)
                if i >= len(block):
                    flight += 1
                    i = flight

            # print("trying block rotation", block)
            # check if this rotation works
            # CHECK: of the first F races, neither team has participated in one of the last F races
            last_row = races[-len(flights) :]
            busy_teams = {r.team1 for r in last_row}.union({r.team2 for r in last_row})
            clash = False
            for flight in range(len(flights)):
                t1, t2 = block[flight]
                if t1 in busy_teams or t2 in busy_teams:
                    clash = True
                    break
                else:
                    pass

                if len(races) > len(flights) - flight - 1:
                    busy_teams.discard(races[flight - len(flights)].team1)
                    busy_teams.discard(races[flight - len(flights)].team2)

            if clash:
                # this block clashes with the previous races
                # rotate races so different block is tried
                first_race = r[-1]
                if len(r) % 2 == 1:
                    first_race = (first_race[1], first_race[0])
                r[0], r[1:] = first_race, r[:-1]
            else:
                # valid block was found
                valid_block_found = True
                break

        if valid_block_found:
            # valid block was found
            print("valid block found")
            for lt, rt in block:
                races.append(Race(lt, rt, len(races) + 1))
            r = rounds.pop(round_num)

            if len(races) > len(best_schedule):
                best_schedule[:] = races

            if block_extend(races, teams, flights, rounds, best_schedule):
                return True
            rounds[round_num] = r
            for _ in block:
                races.pop()

    return False

def greedy_blocks(teams: List[str], flights: List[Flight]) -> List[Race]:
    """
    strategy: at each stage, pick the block which can be built with the least double handovers
    """

    races = []
    best_schedule = []
    rounds = dict(enumerate(generate_rounds(teams)))
    for round_num, r in rounds.items():
        print("round", round_num)
        print(r)

    success = block_extend(races, teams, flights, rounds, best_schedule)

    if success:
        check_schedule(races, teams, flights)
        return races
    else:
        check_schedule(races, teams, flights, check_rr=False)
        print("failed to form blocks")
        return best_schedule

In [21]:

# for pasting into google sheets or excel
def tsv_schedule(teams: List[str], flights: List[Flight], races: List[Race]) -> str:
    # header
    output = (
        "\t" + "\t\t".join(name for name, _, _ in flights) + "\n"
        + "race\t"
        + "\t".join(f"{f1[0]}{f1[1]}\t{f2[0]}{f2[1]}" for _, f1, f2 in flights)
        + "\n"
    )

    # body
    total_raced = {t: 0 for t in teams}
    for race in races:
        total_raced[race.team1] += 1
        total_raced[race.team2] += 1

        output += f"{race.race_num}\t"
        flight_num = (race.race_num - 1) % len(flights)
        output += "\t" * (flight_num * 2)
        output += f"{race.team1}\t{race.team2}"
        # min_raced = min(total_raced.values())
        # max_raced = max(total_raced.values())
        # output += (
        #     "\t" * ((len(flights) - flight_num - 1) * 2)
        #     + f"\t{min_raced}\t{max_raced}\t{max_raced - min_raced}"
        # )
        output += "\n"

        # keep track of how many races each team has raced

        # if all teams have raced the same number of times, add a divider
        # if all(total_raced[t] == total_raced[teams[0]] for t in teams):
        #     output += "breakpoint\n"

    # footer
    return output


In [23]:

BATH_TEAMS: List[Team] = [
    "Reading Red",
    "UCL Purple",
    "Brunel 80085",
    "Manchester Yellow",
    "Imperial Red",
    "Kobra Hurricane",
    "Cardiff Quack",
    "Southampton Blues",
    "Manchester Purple",
    "Sussex Pink",
    "Cambridge Orange",
    "Bristol beige rage",
    "Bristol mellow yellow",
    "Clifton",
    "Bath Freshticles",
    "Bath All-stars",
    "Hybrid Black",
    "Exeter",
]


BATH_FLIGHTS: List[Flight] = [
    ('Clifton', ('Red'        , [1,2,3]   ), ('Blue'       , [4,5,6]   )),
    ('Bristol', ('Blue Stripe', [7,8,9]   ), ('Grey Stripe', [10,11,12])),
    ('Bath',    ('Blue'       , [13,14,15]), ('Yellow'     , [16,17,18])),
]

def bath_quali_to_tsv():
    schedule = greedy_blocks(BATH_TEAMS, BATH_FLIGHTS)
    for r in schedule:
        r.league = "quali"
    with open(f"schedule/bath_quali.tsv", "w") as f:
        f.write(tsv_schedule(BATH_TEAMS, BATH_FLIGHTS, schedule))

def write_tsv_schedule(path: str):
    schedule, flights, teams = tsv_to_schedule(path)
    write_schedule(schedule, 'bathrobe', 1, 'quali')
    team_races = {t: 0 for t in teams}
    for race in schedule:
        team_races[race.team1] += 1
        team_races[race.team2] += 1
    print(team_races)

write_tsv_schedule("./schedule/bath_quali.tsv")
    

{'Southampton Blues': 17, 'Brunel 80085': 17, 'Manchester Yellow': 17, 'Exeter': 17, 'Bath Freshticles': 17, 'Imperial Red': 17, 'Sussex Pink': 17, 'Clifton': 17, 'Bristol beige rage': 17, 'Cardiff Quack': 17, 'Bath All-stars': 17, 'Hybrid Black': 17, 'Reading Red': 17, 'UCL Purple': 17, 'Manchester Purple': 17, 'Kobra Hurricane': 17, 'Cambridge Orange': 17, 'Bristol mellow yellow': 17}
